# Task 3: Citation Span Extraction - BERT Inference

**Model:** bert-base-uncased fine-tuned on citation span extraction

**Input:** `.label` files from test set

**Output:** Predicted span for each citation in each document

**Notes:**
- Dùng softmax độc lập cho start/end (không dùng pipeline vì score underflow)
- Chỉ tìm span trong phần context (sequence_id=1), bỏ qua phần question
- Đảm bảo start <= end

---

## 1. Setup

In [ ]:
import os, json
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print(f"CUDA: {torch.cuda.is_available()}")

## 2. Config

In [ ]:
MODEL_DIR  = "/kaggle/input/task3-bert-citation-span-v2/task3_bert_final_v2"
TEST_DIR   = "/kaggle/input/datasets/tathiyennhi/task3-citation-span-extraction/task3/test"
OUTPUT_DIR = "/kaggle/working/predictions"
MAX_LENGTH = 512
MAX_ANSWER_LEN = 100  # token

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model:  {MODEL_DIR}")
print(f"Test:   {TEST_DIR}")
print(f"Output: {OUTPUT_DIR}")

## 3. Load Model

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

print(f"✅ Model loaded on {DEVICE}")

## 4. Inference Function

In [ ]:
def predict_span(question, context):
    """
    Predict citation span using independent softmax for start/end.
    Returns (pred_text, score, char_start, char_end)
    """
    inputs = tokenizer(
        question, context,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )
    encoded = tokenizer(
        question, context,
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LENGTH
    )
    sequence_ids = encoded.sequence_ids()
    offset_mapping = inputs.pop("offset_mapping")[0]
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_probs = F.softmax(outputs.start_logits[0], dim=0)
    end_probs   = F.softmax(outputs.end_logits[0], dim=0)

    best_score, best_start, best_end = -1, 0, 0
    for s in range(len(start_probs)):
        for e in range(s, min(s + MAX_ANSWER_LEN, len(end_probs))):
            if sequence_ids[s] != 1 or sequence_ids[e] != 1:
                continue
            score = start_probs[s].item() * end_probs[e].item()
            if score > best_score:
                best_score, best_start, best_end = score, s, e

    char_start = offset_mapping[best_start][0].item()
    char_end   = offset_mapping[best_end][1].item()
    pred_text  = context[char_start:char_end]

    return pred_text, best_score, char_start, char_end


print("✅ Inference function defined")

## 5. Run Inference on Test Set

In [ ]:
label_files = sorted(Path(TEST_DIR).glob("*.label"))
print(f"📊 Test files: {len(label_files):,}")

all_predictions = []
skipped = 0

for i, label_file in enumerate(label_files):
    if (i + 1) % 100 == 0:
        print(f"⏳ {i+1:,}/{len(label_files):,}")

    try:
        with open(label_file) as f:
            data = json.load(f)
    except:
        skipped += 1
        continue

    doc_id  = data.get("doc_id", label_file.stem)
    context = data.get("text", "")
    if not context:
        skipped += 1
        continue

    doc_preds = {"doc_id": doc_id, "predictions": []}

    for span_info in data.get("citation_spans", []):
        citation_id = span_info.get("citation_id", "")
        question    = f"What does citation {citation_id} support?"

        pred_text, score, char_start, char_end = predict_span(question, context)

        doc_preds["predictions"].append({
            "citation_id": citation_id,
            "pred_span_text": pred_text,
            "pred_s_span": char_start,
            "pred_e_span": char_end,
            "score": round(score, 6)
        })

    all_predictions.append(doc_preds)

# Save
output_path = f"{OUTPUT_DIR}/predictions.json"
with open(output_path, "w") as f:
    json.dump(all_predictions, f, indent=2)

print(f"\n✅ Done: {len(all_predictions):,} docs | Skipped: {skipped}")
print(f"✅ Saved to: {output_path}")

## 6. Sample Output

In [ ]:
for doc in all_predictions[:2]:
    print(f"\ndoc_id: {doc['doc_id']}")
    for p in doc['predictions']:
        print(f"  {p['citation_id']} | score={p['score']:.4f}")
        print(f"  → {p['pred_span_text'][:100]}...")